In [1]:
import random
import sys
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.sql import SparkSession, functions as F


In [2]:
import os

os.environ["JAVA_HOME"] = r"C:/Program Files/Eclipse Adoptium/jdk-17.0.20.101-hotspot"
os.environ["HADOOP_HOME"] = r"C:/hadoop"

python_path = sys.executable

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

In [3]:
spark = SparkSession.builder.master("local[*]").appName("sparkml-food").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [4]:
bad = ["mice droppings found in storage area", "no hot water at hand sink",
           "food held at unsafe temperature", "dirty food contact surfaces and equipment",
           "evidence of roaches in kitchen", "no certified food manager present"]
ok = ["floors need cleaning", "light shield missing in prep area",
          "wall paint peeling near dishwasher", "thermometer not available"]
rows = []
for _ in range(600):
    if random.random() < 0.5:
        rows.append(("Fail", ". ".join(random.sample(bad, random.randint(2, 4)) + random.sample(ok, 1))))
    else:
        rows.append((random.choice(["Pass", "Pass w/ Conditions"]),
                         ". ".join(random.sample(ok, random.randint(0, 2)))))
raw = spark.createDataFrame(rows, ["Results", "Violations"])

In [5]:
data = (raw.select("Results", "Violations")
           .where(F.col("Results").isin("Pass", "Pass w/ Conditions", "Fail"))
           .withColumn("label", F.when(F.col("Results") == "Fail", 0.0).otherwise(1.0))
           .withColumn("Violations", F.coalesce(F.col("Violations"), F.lit("")))
           .select("Violations", "label"))
print("Rows used:", data.count())
data.groupBy("label").count().show()

train, test = data.randomSplit([0.8, 0.2], seed=42)
print(f"Train rows: {train.count()}   Test rows: {test.count()}\n")

Rows used: 600
+-----+-----+
|label|count|
+-----+-----+
|  0.0|  290|
|  1.0|  310|
+-----+-----+

Train rows: 482   Test rows: 118



In [6]:
pipeline = Pipeline(stages=[
    Tokenizer(inputCol="Violations", outputCol="words"),
    StopWordsRemover(inputCol="words", outputCol="filtered"),
    HashingTF(inputCol="filtered", outputCol="tf", numFeatures=4096),
    IDF(inputCol="tf", outputCol="features"),
    LogisticRegression(maxIter=20, regParam=0.01),
])
model = pipeline.fit(train)
pred = model.transform(test)

pred.select("label", "prediction", "probability").show(8, truncate=60)

+-----+----------+------------------------------------------+
|label|prediction|                               probability|
+-----+----------+------------------------------------------+
|  1.0|       1.0| [0.006455354601805821,0.9935446453981942]|
|  0.0|       0.0| [0.9671738728825517,0.032826127117448256]|
|  0.0|       0.0| [0.9990666816162105,9.333183837895165E-4]|
|  0.0|       0.0| [0.9992535116501092,7.464883498907726E-4]|
|  1.0|       1.0| [0.006368945048023594,0.9936310549519765]|
|  0.0|       0.0|[0.9999854292876155,1.4570712384531781E-5]|
|  0.0|       0.0|[0.9999891322276073,1.0867772392675334E-5]|
|  1.0|       1.0|  [0.014675603390642028,0.985324396609358]|
+-----+----------+------------------------------------------+
only showing top 8 rows


In [7]:
auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC").evaluate(pred)
acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy").evaluate(pred)
f1 = MulticlassClassificationEvaluator(labelCol="label", metricName="f1").evaluate(pred)
print(f"Test accuracy = {acc:.3f} | F1 = {f1:.3f} | AUC-ROC = {auc:.3f}")

print("\nConfusion matrix (label x prediction):")
pred.groupBy("label").pivot("prediction", [0.0, 1.0]).count().na.fill(0).show()
spark.stop()

Test accuracy = 1.000 | F1 = 1.000 | AUC-ROC = 1.000

Confusion matrix (label x prediction):
+-----+---+---+
|label|0.0|1.0|
+-----+---+---+
|  0.0| 61|  0|
|  1.0|  0| 57|
+-----+---+---+

